# Task 3 — Domain Generalization on PACS (sources: Photo, Art Painting, Cartoon; unseen target: Sketch)

* **ERM** = Task 2's Source-only checkpoint, loaded unchanged (never retrained).
* **DAN-DG** (λ_DG = 1): ERM + (λ/3)·Σ over the three source pairs of MMD² — the same MMD code and kernel construction as Task 2's DAN (unbiased U-statistic estimate; see the Task 2 notebook), with bandwidths computed per pair.
* **SAM** (ρ = 0.05): non-adaptive SAM around AdamW; two forward/backward passes per step, BatchNorm statistics frozen in both.
* Same splits (`shared/splits/pacs_sketch_seed6304.json`), network, preprocessing, augmentation, 8-per-source batches, optimizer, epoch budget, early stopping and seed as Task 2 (one shared engine, `shared/pacs_engine.py`).

**Sketch isolation.** Training and all source-side diagnostics receive a split dictionary that *does not contain* Sketch (any access raises a `KeyError`). Sketch is loaded in exactly one place, the "Final Sketch evaluation" section, after the configuration-lock cell. No Task 2 target result was used to choose any Task 3 setting: all settings below are the ones fixed by the assignment.

Controlled design study (Step 5): **DAN-DG with λ_DG ∈ {0.1, 1, 10}** (parallels the Task 2 DAN study, so target-aware and target-free alignment can be compared at matched strength). An optional SAM ρ study can be switched on with `RUN_SAM_RHO_STUDY`.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from common.seed import set_seed
from common.paths import results_dir, checkpoints_dir, figure_path
from common.logging import save_table, save_json
from common.metrics import macro_f1
from shared.pacs import PACS_CLASSES
from shared.pacs_engine import PACSTrainConfig, SOURCE_DOMAINS, load_history

SEED = 6304
QUICK_RUN = False
RETRAIN = False
RUN_SAM_RHO_STUDY = False     # optional second study (not required)
NUM_WORKERS = 2            # per domain loader (4 loaders run concurrently)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS, CKPT = results_dir(3), checkpoints_dir(3)
set_seed(SEED)
pd.set_option("display.precision", 4); pd.set_option("display.width", 220)
cfg = PACSTrainConfig(seed=SEED, device=DEVICE, num_workers=NUM_WORKERS,
                      max_epochs=2 if QUICK_RUN else 30, max_steps_per_epoch=20 if QUICK_RUN else None)

In [2]:
from task3.train import (get_source_splits, train_run, MAIN_RUNS, STUDY_RUNS, STUDY_LAMBDAS, study_run_name,
                         SAM_STUDY_RUNS, SAM_RHOS, sam_run_name, TASK2_ERM_CHECKPOINT)
from task3.methods.erm import load_erm_baseline
source_splits = get_source_splits()
print("domains available to Task 3 training:", list(source_splits))

domains available to Task 3 training: ['photo', 'art_painting', 'cartoon']


## Step 1 — ERM baseline (Task 2 Source-only checkpoint)

In [3]:
models, summaries = {}, {}
models["erm"], summaries["erm"] = load_erm_baseline(TASK2_ERM_CHECKPOINT, DEVICE)
print("ERM checkpoint:", TASK2_ERM_CHECKPOINT, "| selected epoch", summaries["erm"]["best_epoch"])

ERM checkpoint: /workspace/pa1-beyond-iid/task2/checkpoints/source_only.pth | selected epoch 10


## Steps 2–3 — DAN-DG (λ_DG = 1) and SAM (ρ = 0.05)

In [4]:
for name in MAIN_RUNS:
    models[name], summaries[name] = train_run(name, source_splits, cfg, retrain=RETRAIN)

[dan_dg] epoch  0 cls 0.463 extra 0.054 src-train-acc 0.840 mean-val-F1 0.8974


[dan_dg] epoch  1 cls 0.222 extra 0.034 src-train-acc 0.929 mean-val-F1 0.9072


[dan_dg] epoch  2 cls 0.159 extra 0.016 src-train-acc 0.947 mean-val-F1 0.9107


[dan_dg] epoch  3 cls 0.145 extra 0.033 src-train-acc 0.950 mean-val-F1 0.9155


[dan_dg] epoch  4 cls 0.109 extra 0.031 src-train-acc 0.962 mean-val-F1 0.9437


...

[sam] epoch 14 cls 0.010 src-train-acc 0.999 mean-val-F1 0.9523


[sam] epoch 15 cls 0.009 src-train-acc 0.999 mean-val-F1 0.9391


[sam] epoch 16 cls 0.009 src-train-acc 0.999 mean-val-F1 0.9377


[sam] epoch 17 cls 0.007 src-train-acc 0.999 mean-val-F1 0.9431


[sam] epoch 18 cls 0.006 src-train-acc 1.000 mean-val-F1 0.9448


## Step 5 — Controlled design study: DAN-DG with λ_DG ∈ {0.1, 1, 10}
Expected effect of increasing λ_DG (stated before Sketch is seen): source validation performance flat or slightly lower; **source-domain separability lower**; Sketch performance uncertain — invariance to Photo/Art/Cartoon differences may or may not transfer to sketches, and a large λ may remove class-discriminative information. The main comparison keeps λ_DG = 1 and ρ = 0.05 regardless of the outcome.

In [5]:
for name in STUDY_RUNS:
    models[name], summaries[name] = train_run(name, source_splits, cfg, retrain=RETRAIN)
if RUN_SAM_RHO_STUDY:
    for name in SAM_STUDY_RUNS:
        models[name], summaries[name] = train_run(name, source_splits, cfg, retrain=RETRAIN)
display(save_table([{"run": n, "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"],
                     "stopped_early": s.get("stopped_early"), "best_mean_source_val_f1": s["best_mean_val_macro_f1"]}
                    for n, s in summaries.items()], RESULTS / "training_summary"))

[dan_dg_lambda0.1] epoch  0 cls 0.480 extra 0.013 src-train-acc 0.834 mean-val-F1 0.9126


[dan_dg_lambda0.1] epoch  1 cls 0.207 extra 0.008 src-train-acc 0.929 mean-val-F1 0.9178


[dan_dg_lambda0.1] epoch  2 cls 0.144 extra 0.005 src-train-acc 0.952 mean-val-F1 0.9160


[dan_dg_lambda0.1] epoch  3 cls 0.132 extra 0.006 src-train-acc 0.954 mean-val-F1 0.9099


[dan_dg_lambda0.1] epoch  4 cls 0.091 extra 0.005 src-train-acc 0.967 mean-val-F1 0.8803


...

[dan_dg_lambda10] epoch  2 cls 1.941 extra 0.129 src-train-acc 0.266 mean-val-F1 0.1343


[dan_dg_lambda10] epoch  3 cls 1.807 extra 0.119 src-train-acc 0.323 mean-val-F1 0.2833


[dan_dg_lambda10] epoch  4 cls 1.739 extra 0.124 src-train-acc 0.327 mean-val-F1 0.0708


[dan_dg_lambda10] epoch  5 cls 1.939 extra 0.466 src-train-acc 0.206 mean-val-F1 0.1494


,run,best_epoch,epochs_run,stopped_early,best_mean_source_val_f1
0,erm,10,16,True,0.9430
1,dan_dg,4,10,True,0.9437
2,sam,13,19,True,0.9549
3,dan_dg_lambda0.1,6,12,True,0.9403
4,dan_dg_lambda10,0,6,True,0.3779


## Training curves (classification loss, source-validation F1, pairwise MMD² for DAN-DG)

In [6]:
from shared.pacs_analysis import training_curves
t2_results = results_dir(2)
histories = {"erm (T2 source_only)": load_history(t2_results, "source_only")}
histories.update({n: load_history(RESULTS, n) for n in models if n != "erm"})
training_curves({k: histories[k] for k in ["erm (T2 source_only)", "dan_dg", "sam"]}, figure_path("task3_training_curves.png"))
training_curves({n: histories[study_run_name(l)] for l in STUDY_LAMBDAS for n in [study_run_name(l)]},
                figure_path("task3_dan_dg_lambda_training_curves.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task3_dan_dg_lambda_training_curves.png')

## Step 4a — Source-side evaluation and diagnostics (no Sketch)
* Accuracy / macro-F1 per source validation domain, mean and worst.
* Source-domain separability: frozen features of the three source-validation sets, balanced to the smallest domain (seed 6304), stratified 70/30 split, standardized features, multinomial logistic regression with C = 1 (chance 33.3 %).
* Sharpness proxy: one fixed validation batch (32 per source, seed 6304), model in eval mode, Δ = L(θ + 0.05·g/‖g‖) − L(θ). Radii 0.01/0.02/0.1 are reported only as a robustness check of the ranking.

In [7]:
from task3.evaluate_sketch import source_diagnostics
from shared.pacs_analysis import source_rows
RADII = (0.01, 0.02, 0.05, 0.1)
diag, sharp_batch = source_diagnostics(models, source_splits, cfg, radii=RADII, seed=SEED)
save_json(sharp_batch, RESULTS / "sharpness_batch_indices.json")
src_rows = []
for n in models:
    r = {"run": n, **source_rows(diag[n], list(SOURCE_DOMAINS)),
         "source_domain_separability": diag[n]["source_domain_separability"],
         "sharpness_delta_rho0.05": diag[n]["sharpness"][0.05]["delta"],
         "fixed_batch_loss": diag[n]["sharpness"][0.05]["loss"]}
    src_rows.append(r)
src_df = save_table(src_rows, RESULTS / "step4_source_side")
display(src_df)
sharp_rows = [{"run": n, "rho": r, **diag[n]["sharpness"][r]} for n in models for r in RADII]
sharp_df = save_table(sharp_rows, RESULTS / "step4_sharpness_radii")
display(sharp_df.pivot(index="rho", columns="run", values="delta"))
fig, ax = plt.subplots(figsize=(4.2, 3.1))
for n in ["erm", "dan_dg", "sam"]:
    d = sharp_df[sharp_df.run == n].sort_values("rho")
    ax.plot(d.rho, d.delta, "-o", ms=4, label=n)
ax.axvline(0.05, color="gray", ls=":")
ax.set_xlabel("perturbation radius ρ"); ax.set_ylabel("Δ cross-entropy"); ax.set_xscale("log"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.tight_layout()
from common.plotting import save_figure
save_figure(fig, figure_path("task3_sharpness_radii.png"))

,run,photo_acc,photo_f1,art_painting_acc,art_painting_f1,cartoon_acc,cartoon_f1,mean_source_acc,mean_source_f1,worst_source_acc,worst_source_f1,worst_source_domain_by_f1,source_domain_separability,sharpness_delta_rho0.05,fixed_batch_loss
0,erm,0.9610,0.9515,0.9341,0.9305,0.9467,0.9470,0.9473,0.9430,0.9341,0.9305,art_painting,0.8633,0.3176,0.2103
1,dan_dg,0.9760,0.9712,0.9195,0.9177,0.9403,0.9421,0.9453,0.9437,0.9195,0.9177,art_painting,0.7167,0.2997,0.1887
2,sam,0.9640,0.9553,0.9488,0.9490,0.9574,0.9606,0.9567,0.9549,0.9488,0.9490,art_painting,0.8433,0.0911,0.0996
3,dan_dg_lambda0.1,0.9580,0.9521,0.9293,0.9309,0.9339,0.9377,0.9404,0.9403,0.9293,0.9309,art_painting,0.8667,0.3533,0.2047
4,dan_dg_lambda10,0.5526,0.4391,0.3463,0.2889,0.4072,0.4057,0.4354,0.3779,0.3463,0.2889,art_painting,0.6867,2.2741,1.4972


run,dan_dg,dan_dg_lambda0.1,dan_dg_lambda10,erm,sam
rho,,,,,
0.01,0.0539,0.0660,0.2549,0.0593,0.0167
0.02,0.1113,0.1370,0.6411,0.1218,0.0342
0.05,0.2997,0.3533,2.2741,0.3176,0.0911
0.10,0.7070,0.6645,5.8756,0.6567,0.1950


PosixPath('/workspace/pa1-beyond-iid/report/figures/task3_sharpness_radii.png')

In [8]:
# Representation diagnostics on the source-validation features (no Sketch). Class information that survives
# alignment is measured with a 5-fold linear class probe; collapse shows up as a small feature norm, many exact
# zeros and a low effective rank (exp of the entropy of the normalized singular-value spectrum).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
feat_rows = []
for n in models:
    Fv = np.concatenate([diag[n][d]["features"] for d in SOURCE_DOMAINS])
    yv = np.concatenate([diag[n][d]["labels"] for d in SOURCE_DOMAINS])
    s = np.linalg.svd(Fv - Fv.mean(0), compute_uv=False)
    p = s ** 2 / (s ** 2).sum()
    p = p[p > 0]
    probe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000))
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    feat_rows.append({"run": n, "mean_feature_norm": float(np.linalg.norm(Fv, axis=1).mean()),
                      "frac_zero_activations": float((Fv == 0).mean()),
                      "effective_rank": float(np.exp(-(p * np.log(p)).sum())),
                      "class_probe_acc_5fold": float(cross_val_score(probe, Fv, yv, cv=cv).mean()),
                      "mean_source_f1": [r for r in src_rows if r["run"] == n][0]["mean_source_f1"]})
display(save_table(feat_rows, RESULTS / "step4_feature_diagnostics"))

,run,mean_feature_norm,frac_zero_activations,effective_rank,class_probe_acc_5fold,mean_source_f1
0,erm,70.8267,0.0514,11.7247,0.9530,0.9430
1,dan_dg,41.3207,0.0284,16.0711,0.9406,0.9437
2,sam,42.1803,0.2150,12.9543,0.9530,0.9549
3,dan_dg_lambda0.1,54.9962,0.0376,14.5820,0.9489,0.9403
4,dan_dg_lambda10,106.6458,0.3018,2.6529,0.5619,0.3779


## Configuration lock
Every Task 3 checkpoint and setting is now fixed. Sketch is loaded for the first time below.

In [9]:
save_json({"config": {k: str(v) for k, v in cfg.__dict__.items()}, "runs": summaries}, RESULTS / "locked_configuration.json")

PosixPath('/workspace/pa1-beyond-iid/task3/results/locked_configuration.json')

## Step 4b — Final Sketch evaluation

In [10]:
from task2.train import get_splits as get_full_splits
from task3.evaluate_sketch import sketch_predictions
full_splits = get_full_splits()
sk = sketch_predictions(models, full_splits, cfg)
rows = []
for n in models:
    t = sk[n]
    rows.append({**[r for r in src_rows if r["run"] == n][0],
                 "sketch_acc": float((t["preds"] == t["labels"]).mean()), "sketch_f1": macro_f1(t["labels"], t["preds"])})
erm_acc = rows[0]["sketch_acc"]
for r in rows:
    r["sketch_acc_change_vs_erm"] = r["sketch_acc"] - erm_acc
final = save_table(rows, RESULTS / "step4_final_comparison")
cols = ["run"] + [f"{d}_acc" for d in SOURCE_DOMAINS] + [f"{d}_f1" for d in SOURCE_DOMAINS] + \
       ["mean_source_acc", "mean_source_f1", "worst_source_acc", "worst_source_f1", "sketch_acc", "sketch_f1",
        "sketch_acc_change_vs_erm", "source_domain_separability", "sharpness_delta_rho0.05"]
display(final[final.run.isin(["erm", "dan_dg", "sam"])][cols])
np.savez_compressed(RESULTS / "sketch_predictions.npz", labels=sk["erm"]["labels"],
                    **{f"{n}__preds": sk[n]["preds"] for n in models})

,run,photo_acc,art_painting_acc,cartoon_acc,photo_f1,art_painting_f1,cartoon_f1,mean_source_acc,mean_source_f1,worst_source_acc,worst_source_f1,sketch_acc,sketch_f1,sketch_acc_change_vs_erm,source_domain_separability,sharpness_delta_rho0.05
0,erm,0.961,0.9341,0.9467,0.9515,0.9305,0.9470,0.9473,0.9430,0.9341,0.9305,0.5719,0.6020,0.0000,0.8633,0.3176
1,dan_dg,0.976,0.9195,0.9403,0.9712,0.9177,0.9421,0.9453,0.9437,0.9195,0.9177,0.5849,0.6360,0.0130,0.7167,0.2997
2,sam,0.964,0.9488,0.9574,0.9553,0.9490,0.9606,0.9567,0.9549,0.9488,0.9490,0.7422,0.7597,0.1703,0.8433,0.0911


In [11]:
# Do source-side rankings predict Sketch? (rank correlations over all Task 3 runs)
from scipy.stats import spearmanr
for col in ["mean_source_f1", "worst_source_f1", "source_domain_separability", "sharpness_delta_rho0.05"]:
    rho_, p_ = spearmanr(final[col], final["sketch_acc"])
    print(f"Spearman({col}, sketch_acc) = {rho_:+.2f} (p={p_:.2f}, n={len(final)})")

Spearman(mean_source_f1, sketch_acc) = +0.70 (p=0.19, n=5)
Spearman(worst_source_f1, sketch_acc) = +0.90 (p=0.04, n=5)
Spearman(source_domain_separability, sketch_acc) = +0.50 (p=0.39, n=5)
Spearman(sharpness_delta_rho0.05, sketch_acc) = -0.70 (p=0.19, n=5)


### Class-level analysis and comparison with Task 2

In [12]:
from shared.pacs_analysis import (per_class_delta_rows, largest_change_rows, transition_counts, confusion_grid,
                                  per_class_delta_plot, failure_grid)
main = {n: sk[n] for n in ["erm", "dan_dg", "sam"]}
pc = save_table(per_class_delta_rows(main, "erm"), RESULTS / "step4_per_class_sketch")
display(pc.pivot(index="class", columns="run", values="accuracy"))
display(save_table(transition_counts(main, "erm"), RESULTS / "step4_fixed_vs_broken"))
changes = largest_change_rows(main, "erm")
save_json(changes, RESULTS / "step4_largest_changes_confusions.json")
for c in changes:
    print(f"{c['run']:7s} {c['kind']:14s} {c['class']:9s} delta={c['delta']:+.3f} | run: "
          + ", ".join(f"{x['predicted_as']} {x['fraction_of_class']:.2f}" for x in c["confusions_run"])
          + " | ERM: " + ", ".join(f"{x['predicted_as']} {x['fraction_of_class']:.2f}" for x in c["confusions_baseline"]))
per_class_delta_plot(pc.to_dict("records"), "erm", figure_path("task3_per_class_delta.png"), ylabel="Sketch acc. change vs. ERM")
confusion_grid(main, figure_path("task3_sketch_confusions.png"))

# Task 2 (target-aware) vs Task 3 (target-free), same ERM baseline, same Sketch images and order.
t2 = np.load(results_dir(2) / "target_predictions.npz", allow_pickle=True)
assert np.array_equal(t2["labels"], sk["erm"]["labels"])
print("ERM vs Task 2 Source-only prediction agreement on Sketch:", float((t2["source_only__preds"] == sk["erm"]["preds"]).mean()))
cross = {"erm": sk["erm"], "dan (T2, sees Sketch)": {"preds": t2["dan__preds"], "labels": t2["labels"]},
         "dan_dg (T3)": sk["dan_dg"], "sam (T3)": sk["sam"]}
for n in ["dann", "cdan"]:
    cross[f"{n} (T2)"] = {"preds": t2[f"{n}__preds"], "labels": t2["labels"]}
cpc = save_table(per_class_delta_rows(cross, "erm"), RESULTS / "step4_per_class_task2_vs_task3")
display(cpc.pivot(index="class", columns="run", values="accuracy"))
per_class_delta_plot(cpc.to_dict("records"), "erm", figure_path("task3_vs_task2_per_class_delta.png"), ylabel="Sketch acc. change vs. ERM")
ov = [{"run": n, "sketch_acc": float((p["preds"] == p["labels"]).mean()), "sketch_f1": macro_f1(p["labels"], p["preds"])} for n, p in cross.items()]
display(save_table(ov, RESULTS / "step4_task2_vs_task3_overall"))
# DAN vs DAN-DG at matched lambda (both studies)
t2_study = pd.read_csv(results_dir(2) / "step6_dan_lambda_study.csv")
matched = []
for l in STUDY_LAMBDAS:
    a = t2_study[t2_study.lambda_mmd == l].iloc[0]
    b = final[final.run == study_run_name(l)].iloc[0]
    matched.append({"lambda": l, "DAN_T2_sketch_acc": a.target_acc, "DAN_DG_T3_sketch_acc": b.sketch_acc,
                    "DAN_T2_mean_source_acc": a.mean_source_acc, "DAN_DG_T3_mean_source_acc": b.mean_source_acc})
display(save_table(matched, RESULTS / "step4_dan_vs_dandg_matched_lambda"))

run,dan_dg,erm,sam
class,,,
dog,0.3627,0.4870,0.3303
elephant,0.3230,0.7932,0.9365
giraffe,0.5139,0.8088,0.7477
guitar,0.7796,0.7204,0.9079
horse,0.9375,0.1348,0.8162
house,0.9375,0.9250,0.8875
person,0.4875,0.3312,0.7250


,run,fixed,broken,net,n
0,dan_dg,917,866,51,3929
1,sam,972,303,669,3929


dan_dg  most_improved  horse     delta=+0.803 | run: dog 0.03, giraffe 0.02, elephant 0.01 | ERM: giraffe 0.56, dog 0.21, elephant 0.09
dan_dg  most_degraded  elephant  delta=-0.470 | run: horse 0.50, dog 0.13, giraffe 0.05 | ERM: giraffe 0.10, dog 0.10
sam     most_improved  horse     delta=+0.681 | run: elephant 0.08, giraffe 0.07, dog 0.02 | ERM: giraffe 0.56, dog 0.21, elephant 0.09
sam     most_degraded  dog       delta=-0.157 | run: horse 0.30, elephant 0.26, giraffe 0.07 | ERM: elephant 0.27, giraffe 0.24, person 0.01


ERM vs Task 2 Source-only prediction agreement on Sketch: 1.0


run,cdan (T2),"dan (T2, sees Sketch)",dan_dg (T3),dann (T2),erm,sam (T3)
class,,,,,,
dog,0.6826,0.5453,0.3627,0.0816,0.4870,0.3303
elephant,0.8676,0.6770,0.3230,0.0027,0.7932,0.9365
giraffe,0.5246,0.7955,0.5139,0.9655,0.8088,0.7477
guitar,0.8273,0.9704,0.7796,0.2829,0.7204,0.9079
horse,0.5453,0.7806,0.9375,0.0110,0.1348,0.8162
house,1.0000,1.0000,0.9375,0.0000,0.9250,0.8875
person,0.1187,0.9375,0.4875,0.0000,0.3312,0.7250


,run,sketch_acc,sketch_f1
0,erm,0.5719,0.6020
1,"dan (T2, sees Sketch)",0.7580,0.7322
2,dan_dg (T3),0.5849,0.6360
3,sam (T3),0.7422,0.7597
4,dann (T2),0.2476,0.1329
5,cdan (T2),0.6645,0.5806


,lambda,DAN_T2_sketch_acc,DAN_DG_T3_sketch_acc,DAN_T2_mean_source_acc,DAN_DG_T3_mean_source_acc
0,0.1,0.7216,0.6063,0.9416,0.9404
1,1.0,0.7580,0.5849,0.9298,0.9453
2,10.0,0.6058,0.2268,0.8971,0.4354


In [13]:
# Failure cases: Sketch images broken / fixed by DAN-DG and SAM relative to ERM (examined only now, for analysis).
rng = np.random.RandomState(SEED)
spaths = np.array(full_splits["sketch"]["paths"])
paths_, titles_, fail_rows = [], [], []
base = sk["erm"]
for n in ["dan_dg", "sam"]:
    p = sk[n]
    for kind, mask in [("broken", (base["preds"] == base["labels"]) & (p["preds"] != p["labels"])),
                       ("fixed", (base["preds"] != base["labels"]) & (p["preds"] == p["labels"]))]:
        for i in rng.permutation(np.where(mask)[0])[:4]:
            paths_.append(spaths[i])
            titles_.append(f"{n} {kind}\ntrue {PACS_CLASSES[p['labels'][i]]}\nERM:{PACS_CLASSES[base['preds'][i]]} -> {PACS_CLASSES[p['preds'][i]]}")
            fail_rows.append({"run": n, "kind": kind, "path": spaths[i], "true": PACS_CLASSES[p["labels"][i]],
                              "erm_pred": PACS_CLASSES[base["preds"][i]], "run_pred": PACS_CLASSES[p["preds"][i]]})
# Images every model gets wrong
allwrong = np.where(np.all([sk[n]["preds"] != sk[n]["labels"] for n in ["erm", "dan_dg", "sam"]], 0))[0]
for i in rng.permutation(allwrong)[:8]:
    paths_.append(spaths[i])
    titles_.append(f"all wrong\ntrue {PACS_CLASSES[base['labels'][i]]}\nERM/DG/SAM: " +
                   "/".join(PACS_CLASSES[sk[n]['preds'][i]][:5] for n in ["erm", "dan_dg", "sam"]))
save_table(fail_rows, RESULTS / "step4_failure_examples")
failure_grid(paths_, titles_, figure_path("task3_failure_examples.png"), n_cols=8)

PosixPath('/workspace/pa1-beyond-iid/report/figures/task3_failure_examples.png')

## Step 5 — Controlled study results

In [14]:
from shared.pacs_analysis import study_plot
study = [dict(r, lambda_dg=l) for l in STUDY_LAMBDAS for r in rows if r["run"] == study_run_name(l)]
study_df = save_table(study, RESULTS / "step5_dan_dg_lambda_study")
display(study_df[["lambda_dg", "mean_source_acc", "mean_source_f1", "worst_source_f1", "source_domain_separability",
                  "sharpness_delta_rho0.05", "sketch_acc", "sketch_f1", "sketch_acc_change_vs_erm"]])
study_plot(study_df, "lambda_dg", {"Source validation": {"mean_source_acc": "mean acc", "worst_source_f1": "worst F1"},
                                   "Source-domain separability": {"source_domain_separability": "probe acc (chance .333)"},
                                   "Sketch (unseen)": {"sketch_acc": "accuracy", "sketch_f1": "macro-F1"}},
           figure_path("task3_dan_dg_lambda_study.png"), "λ_DG")
if RUN_SAM_RHO_STUDY:
    sam_study = [dict(r, rho=x) for x in SAM_RHOS for r in rows if r["run"] == sam_run_name(x)]
    sdf = save_table(sam_study, RESULTS / "step5_sam_rho_study")
    display(sdf[["rho", "mean_source_acc", "sharpness_delta_rho0.05", "sketch_acc"]])

,lambda_dg,mean_source_acc,mean_source_f1,worst_source_f1,source_domain_separability,sharpness_delta_rho0.05,sketch_acc,sketch_f1,sketch_acc_change_vs_erm
0,0.1,0.9404,0.9403,0.9309,0.8667,0.3533,0.6063,0.6061,0.0344
1,1.0,0.9453,0.9437,0.9177,0.7167,0.2997,0.5849,0.6360,0.0130
2,10.0,0.4354,0.3779,0.2889,0.6867,2.2741,0.2268,0.2462,-0.3451


## Research questions (answer in your own words)
1. How well do mean-/worst-source validation predict Sketch; which class failures are invisible in aggregate source results? → `step4_final_comparison`, `step4_per_class_sketch`.
2. Does DAN-DG reduce source-domain separability, and does that help Sketch? → separability column, λ study, per-class deltas.
3. Does SAM reduce the sharpness proxy; does the sharpness ranking match the Sketch ranking? → `step4_sharpness_radii`.
4. Target-aware DAN (Task 2) vs. target-free DAN-DG (Task 3) → `step4_per_class_task2_vs_task3`, `step4_dan_vs_dandg_matched_lambda`.